# TCGA-KIRC Preprocessing — RNA-seq + Methylation + Clinical Phenotypes

improvement the previous script: variance-filter FIRST, then impute only the ~5,000 columns actually being kept, cutting the imputation workload by roughly 98%. 

**Layers handled:**
1. `HiSeqV2`- gene expression (RNA-seq)
2. `Methylation450K`- DNA methylation (450K array)
3. `PhenotypeCuratedClinicalDataSurvival`- survival endpoints (OS, OS.time)
4. `Phenotypes`- general clinical covariates (stage, grade, age, gender, etc.)

## 1. Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 20)


## 2. Configuration


In [ ]:
RNASEQ_FILE      = "HiSeqV2.txt"
METHYLATION_FILE = "Methylation450K.txt"
SURVIVAL_FILE    = "PhenotypeCuratedClinicalDataSurvival.txt"
PHENOTYPE_FILE   = "Phenotypes.txt"

OUTPUT_DIR = "preprocessed_multiomics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TUMOR_SUFFIX = "-01"

ENDPOINT_EVENT = "OS"
ENDPOINT_TIME  = "OS.time"

N_TOP_EXPRESSION_GENES   = 5000
N_TOP_METHYLATION_PROBES = 5000
MAX_MISSING_FRACTION     = 0.1  

N_FOLDS = 5
RANDOM_STATE = 42

CLINICAL_COVARIATES_TO_KEEP = []


## 3. Helper functions

In [ ]:
def load_omics_layer(path):
    """Load a Xena-style features x samples file, transpose to samples x features."""
    df = pd.read_csv(path, sep="\t", index_col=0)
    df = df.transpose()
    df.index.name = "sample"
    return df


def restrict_to_tumor_samples(df, suffix=TUMOR_SUFFIX):
    tumor_ids = [s for s in df.index if s.endswith(suffix)]
    return df.loc[tumor_ids]


def clean_and_filter_methylation(df, max_missing_fraction, n_top):
    #FIXED ORDER
    t0 = time.time()

    # 1. Drop probes with too much missing data
    missing_frac = df.isna().mean(axis=0)
    df = df.loc[:, missing_frac <= max_missing_fraction]
    print(f"  kept {df.shape[1]} probes after missing-value filter "
          f"({time.time()-t0:.1f}s so far)")

    # 2. Force numeric dtype 
    n_before = df.shape[1]
    df = df.apply(pd.to_numeric, errors="coerce")
    print(f"  coerced to numeric dtype ({time.time()-t0:.1f}s so far)")

    # 3. Variance-rank on the SURVIVING probes 
    variances = df.var(axis=0, skipna=True).sort_values(ascending=False)
    top_probes = variances.head(n_top).index
    df = df[top_probes]
    print(f"  reduced to top {n_top} variable probes ({time.time()-t0:.1f}s so far)")

    # 4. Impute ONLY these n_top columns
    if df.isna().any().any():
        df = df.fillna(df.mean(axis=0))
        print(f"  imputed remaining missing values on filtered subset only "
              f"({time.time()-t0:.1f}s total)")
    else:
        print(f"  no remaining missing values to impute ({time.time()-t0:.1f}s total)")

    return df


def variance_filter(df, n_top):
    if n_top is None or n_top >= df.shape[1]:
        return df
    variances = df.var(axis=0).sort_values(ascending=False)
    top_features = variances.head(n_top).index
    return df[top_features]


def load_survival(path, event_col, time_col):
    clin = pd.read_csv(path, sep="\t", dtype=str)
    clin = clin.set_index("sample")
    clin = clin[[event_col, time_col]].dropna()
    clin = clin[(clin[event_col] != "") & (clin[time_col] != "")]
    clin[event_col] = clin[event_col].astype(int)
    clin[time_col] = clin[time_col].astype(float)
    return clin


def load_phenotypes(path, covariates_to_keep):
    df = pd.read_csv(path, sep="\t", dtype=str)
    if "sample" in df.columns:
        df = df.set_index("sample")
    else:
        df = df.set_index(df.columns[0])

    print(f"Phenotype file loaded: {df.shape[0]} samples x {df.shape[1]} columns")
    print("Available columns:")
    for c in df.columns:
        print(f"  - {c}")



def make_cv_folds(sample_ids, event_labels, n_folds, random_state):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    fold_assignment = pd.Series(index=sample_ids, dtype=int, name="fold")
    for fold_idx, (_, test_idx) in enumerate(skf.split(sample_ids, event_labels)):
        fold_assignment.iloc[test_idx] = fold_idx
    return fold_assignment


## 4. Load RNA-seq (HiSeqV2)

In [ ]:
expr = load_omics_layer(RNASEQ_FILE)
print("raw shape (samples x genes):", expr.shape)

expr = restrict_to_tumor_samples(expr)
print("after tumor-only filter:", expr.shape)

expr.iloc[:5, :5]


## 5. Load methylation (Methylation450K) - FIXED, fast version

In [ ]:
meth = load_omics_layer(METHYLATION_FILE)
print("raw shape (samples x probes):", meth.shape)

meth = restrict_to_tumor_samples(meth)
print("after tumor-only filter:", meth.shape)

meth_filtered = clean_and_filter_methylation(meth, MAX_MISSING_FRACTION, N_TOP_METHYLATION_PROBES)
print("\nFinal methylation shape:", meth_filtered.shape)
meth_filtered.iloc[:5, :5]


## 6. Load survival endpoints

In [ ]:
clin_outcome = load_survival(SURVIVAL_FILE, ENDPOINT_EVENT, ENDPOINT_TIME)
print(f"samples with valid {ENDPOINT_EVENT}/{ENDPOINT_TIME}:", clin_outcome.shape[0])
clin_outcome.head()


## 7. Load clinical phenotypes

In [ ]:
clin_covariates = load_phenotypes(PHENOTYPE_FILE, CLINICAL_COVARIATES_TO_KEEP)
clin_covariates.head()

## 8. Find samples common to ALL layers

In [ ]:
common_samples = set(expr.index) & set(meth_filtered.index) & set(clin_outcome.index)
if not clin_covariates.empty:
    common_samples &= set(clin_covariates.index)
common_samples = sorted(common_samples)

print(f"Samples common to ALL layers: {len(common_samples)}")
print(f"  (expression alone: {expr.shape[0]}, methylation alone: {meth_filtered.shape[0]})")

expr = expr.loc[common_samples]
meth_filtered = meth_filtered.loc[common_samples]
clin_outcome = clin_outcome.loc[common_samples]
if not clin_covariates.empty:
    clin_covariates = clin_covariates.loc[common_samples]


## 9. Variance filtering (RNA-seq)

In [ ]:
expr_filtered = variance_filter(expr, N_TOP_EXPRESSION_GENES)
print("RNA-seq filtered shape:", expr_filtered.shape)


## 10. Save outputs

In [ ]:
expr_filtered.to_csv(os.path.join(OUTPUT_DIR, "expression_filtered.csv"))
meth_filtered.to_csv(os.path.join(OUTPUT_DIR, "methylation_filtered.csv"))
clin_outcome.to_csv(os.path.join(OUTPUT_DIR, "clinical_outcome.csv"))
folds.to_csv(os.path.join(OUTPUT_DIR, "cv_folds.csv"), header=True)
if not clin_covariates.empty:
    clin_covariates.to_csv(os.path.join(OUTPUT_DIR, "clinical_covariates.csv"))

print("Saved to:", OUTPUT_DIR)
print("  expression_filtered.csv   -> samples x top-variance genes")
print("  methylation_filtered.csv  -> samples x top-variance probes")
print("  clinical_outcome.csv      -> sample, OS, OS.time")
print("  cv_folds.csv              -> sample, fold")
if not clin_covariates.empty:
    print("  clinical_covariates.csv   -> selected phenotype columns")



**Next step:** run `00_build_reference_cohort.ipynb` pointed at this notebook's `preprocessed_multiomics/` output, then proceed to `01_model_expression.ipynb`